# YOLOv11-Seg for TWSI Segmentation
---

This notebook trains YOLOv11-Seg models (Nano and Extra-large variants) for tactile walking
surface indicator instance segmentation using the Ultralytics API.

YOLOv11-Seg provides real-time instance segmentation with pre-trained weights fine-tuned
on the GuideTWSI dataset at 640×640 resolution.

**Paper Reference:** GuideTWSI (ICRA 2026), Section VI-A

## 1. Setup

In [ ]:
!pip install ultralytics>=8.3.0 -q

## 2. Configuration

Load training hyperparameters from the config file.

In [ ]:
import yaml

# Choose variant: yolov11_seg_n.yaml or yolov11_seg_x.yaml
CONFIG_PATH = "../configs/yolov11_seg_n.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

VARIANT = config["model"]["variant"]  # yolov11n-seg or yolov11x-seg
EPOCHS = config["training"]["epochs"]  # 100
BATCH_SIZE = config["training"]["batch_size"]  # 16
IMAGE_SIZE = config["training"]["image_size"]  # 640
LR = config["training"]["learning_rate"]  # 0.01
MOMENTUM = config["training"]["momentum"]  # 0.937

print(f"Model: {VARIANT}")
print(f"Training: epochs={EPOCHS}, batch={BATCH_SIZE}, imgsz={IMAGE_SIZE}, lr={LR}")

## 3. Download Dataset

Download the GuideTWSI dataset from HuggingFace Hub.

The dataset should be in YOLO format with a `data.yaml` file.

In [ ]:
# Download dataset from HuggingFace
# !huggingface-cli download guidedogrobot-tactile/GuideTWSI --local-dir ./data

# Path to YOLO data.yaml
DATA_YAML = "data/data.yaml"  # Update with your actual path

## 4. Training

Fine-tune YOLOv11-Seg with pre-trained weights using SGD optimizer.

**Hyperparameters from paper:**
- SGD with lr=0.01, momentum=0.937
- 100 epochs, batch size 16, 640×640 resolution
- Standard augmentations: mosaic, flips, color jitter

In [ ]:
from ultralytics import YOLO

# Load pre-trained model
model = YOLO(f"{VARIANT}.pt")

# Train
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    lr0=LR,
    momentum=MOMENTUM,
    weight_decay=config["training"]["weight_decay"],
    mosaic=config["augmentation"]["mosaic"],
    fliplr=config["augmentation"]["flip_lr"],
    hsv_h=config["augmentation"]["hsv_h"],
    hsv_s=config["augmentation"]["hsv_s"],
    hsv_v=config["augmentation"]["hsv_v"],
    project="runs/segment",
    name=VARIANT,
)

## 5. Inference

Run inference on test images and visualize predictions.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Load best weights
best_weights = f"runs/segment/{VARIANT}/weights/best.pt"
model = YOLO(best_weights)

# Run inference on sample images
test_dir = "data/test/images"  # Update with your test path
test_images = sorted(os.listdir(test_dir))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, (ax, img_name) in enumerate(zip(axes.flat, test_images)):
    img_path = os.path.join(test_dir, img_name)
    results = model(img_path, verbose=False)

    # Plot result with masks
    result_img = results[0].plot()
    ax.imshow(result_img[..., ::-1])  # BGR -> RGB
    ax.set_title(img_name, fontsize=8)
    ax.axis("off")

plt.suptitle(f"YOLOv11-Seg ({VARIANT}) Predictions", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Evaluation

Evaluate the model using Ultralytics built-in validation and our custom metrics.

In [ ]:
# Ultralytics built-in validation
val_results = model.val(data=DATA_YAML, split="test")
print(f"\nUltralytics Metrics:")
print(f"  mAP50:    {val_results.seg.map50:.4f}")
print(f"  mAP50-95: {val_results.seg.map:.4f}")

In [ ]:
import sys
from tqdm import tqdm

sys.path.insert(0, "..")
from evaluation.metrics import compute_binary_metrics, aggregate_metrics

# Custom binary metrics evaluation
test_img_dir = "data/test/images"
test_mask_dir = "data/test/masks"
all_metrics = []

for img_name in tqdm(sorted(os.listdir(test_img_dir)), desc="Evaluating"):
    if not img_name.endswith((".jpg", ".png")):
        continue

    img_path = os.path.join(test_img_dir, img_name)
    mask_path = os.path.join(test_mask_dir, img_name.replace(".jpg", ".png"))
    if not os.path.exists(mask_path):
        continue

    gt_mask = np.array(Image.open(mask_path).convert("L"))
    results = model(img_path, verbose=False)

    pred_mask = np.zeros_like(gt_mask, dtype=np.uint8)
    if results[0].masks is not None:
        for mask in results[0].masks.data:
            m = mask.cpu().numpy()
            m_resized = np.array(
                Image.fromarray((m * 255).astype(np.uint8)).resize(
                    (gt_mask.shape[1], gt_mask.shape[0])
                )
            )
            pred_mask = np.maximum(pred_mask, m_resized)

    metrics = compute_binary_metrics(gt_mask, pred_mask)
    all_metrics.append(metrics)

avg = aggregate_metrics(all_metrics)
print(f"\nBinary Segmentation Results ({len(all_metrics)} images):")
for k, v in avg.items():
    print(f"  {k}: {v:.4f}")